# Le trajet derrière la matrice de richesse — distillation sans corpus (#1605)

**Candidate pour le chapitre manquant** de la série `Argument_Analysis` (CoursIA).
Le notebook `Argument_Analysis_Formal_Richness_Matrix.ipynb` enseigne les quatre
classes de **verdict** (`substantive` / `honest-absent` / `unavailable` / `theatre`)
sur une démo synthétique de 24 cellules. Le chapitre manquant applique le même
instrument **à l'échelle d'un vrai pipeline de 40 phases** — et la mesure à
l'échelle a forcé un second axe : le **trajet** que parcourt chaque sortie
(appel → sortie → lecture → décision).

Ce notebook dérive la classification par trajet sur un état synthétique.
Pas de corpus, pas de clé API, pas de LLM, pas d'exécution du pipeline.
**C'est l'instrument, pas la mesure** : les chiffres par phase du vrai pipeline
vivent dans les artefacts de campagne (gitignorés) et dans le fil de l'issue ;
voir `docs/coursia_contrib/richness_matrix_at_scale_note.md`.

In [ ]:
import sys
sys.path.insert(0, "/absolute/path/to/2025-Epita-Intelligence-Symbolique")

# The surface the matrix reads: the production state class, importable
# corpus-free (no JVM, no LLM — the containers are plain attributes).
from argumentation_analysis.core.shared_state import UnifiedAnalysisState

_probe = UnifiedAnalysisState("Texte synthetique public. Il ne contient aucun corpus.")
_probe.add_argument("Le projet depasse les bornes budgetaires.")
REAL_CONTAINERS = [
    k for k in ("identified_arguments", "identified_fallacies",
                "modal_analysis_results", "fol_analysis_results",
                "propositional_analysis_results")
    if hasattr(_probe, k)
]
print("Conteneurs reels accessibles sans JVM :", REAL_CONTAINERS)

## 1. Les deux axes

**Axe verdict** (FP-23) — ce que la cellule a *réellement produit* :
`substantive` (a décidé), `honest-absent` (câblée, vide **et** le dit — le corpus
manque la structure), `unavailable` (pas de writer distinct / solveur
injoignable), `theatre` (verdict affirmé sans décision derrière). Les conditions
se lisent sur la **sortie**, jamais sur le câblage — *wiring ≠ output*.

**Axe trajet** (apparu à l'échelle) — jusqu'où la sortie voyage :

| étape | question | preuve |
|---|---|---|
| **appel** | la phase a-t-elle tourné ? | elle figure dans les phases exécutées du run |
| **sortie** | a-t-elle écrit dans l'état ? | la clé d'état visée est peuplée après le run |
| **lecture** | un lecteur du **chemin de la conclusion** la lit-il ? | un lecteur vivant accède à la clé |
| **décision** | ce qui est lu change-t-il un verdict ? | le verdict diffère quand la valeur diffère |

La leçon de l'échelle : **une étape atteinte n'implique pas la suivante**.
Une phase peut être appelée et ne rien écrire (théâtre muet) ; écrire et n'être
jamais lue (attesté non mobilisé) ; être lue sur le chemin de la conclusion et
ne rien peser (« décide sans peser » — mesuré sur l'axe modal, qui atteint le
lecteur via l'annexe mais n'a pas d'entrée dans la bande de verdict de l'Acte III).

In [ ]:
# Reference implementation of the journey classification — transparent on
# purpose (~30 lines). The production measurements instrument REAL runs; this
# pedagogical version classifies a declared toy (next cell).

def stage_appel(phase, run):
    """The phase appears among the run's executed phases."""
    return phase["name"] in run["executed_phases"]


def stage_sortie(phase, state):
    """The state key the writer targets is populated after the run."""
    key = phase["writes"]
    return key is not None and bool(state.get(key))


def stage_lecture(phase, reader_graph):
    """A reader ON THE CONCLUSION PATH reads that key.

    An off-path reader (e.g. an export-only consumer) does NOT count:
    it makes the loss visible, it does not carry the output to the verdict.
    """
    return phase["writes"] in {
        r["reads"] for r in reader_graph if r["on_conclusion_path"]
    }


def stage_decision(phase, state, reader_graph, verdict_fn):
    """Substitution test: the verdict changes when the output is emptied.

    The same test the campaign applies to its gates — a classifier whose
    output cannot change under substitution measures nothing.
    """
    key = phase["writes"]
    if key is None or not stage_lecture(phase, reader_graph):
        return False
    muted = {k: v for k, v in state.items() if k != key}
    return verdict_fn(state) != verdict_fn(muted)


def journey(phase, run, state, reader_graph, verdict_fn):
    """Ordered stop-point: each stage requires the previous one."""
    if not stage_appel(phase, run):
        return []
    steps = ["appel"]
    if not stage_sortie(phase, state):
        return steps
    steps.append("sortie")
    if not stage_lecture(phase, reader_graph):
        return steps
    steps.append("lecture")
    if stage_decision(phase, state, reader_graph, verdict_fn):
        steps.append("decision")
    return steps

In [ ]:
# Toy pipeline: five SYNTHETIC phases over container names that mirror the
# real state surface (identified_arguments, identified_fallacies,
# modal_analysis_results are real attributes — cf. the probe above). Phase
# names are synthetic on purpose, and the fourth key is explicitly synthetic:
# WHICH real key is written-but-never-read is an empirical result (measured
# historically in the issue thread), not something this toy asserts.
# Nothing here claims a real phase's class; the toy shows HOW each
# stop-point is derived.

TOY_PHASES = [
    # archetype: full journey (extraction, fallacy detection — the measured carriers)
    {"name": "p_extract",  "writes": "identified_arguments"},
    {"name": "p_fallacy",  "writes": "identified_fallacies"},
    # archetype: read on the conclusion path, never weighs (the modal case, R752)
    {"name": "p_formal_a", "writes": "modal_analysis_results"},
    # archetype: written, read only OFF the conclusion path
    {"name": "p_formal_b", "writes": "synth_unread_output"},
    # archetype: called, writes nothing (silent decor)
    {"name": "p_decor",    "writes": None},
]

TOY_RUN = {"executed_phases": {p["name"] for p in TOY_PHASES}}

TOY_READERS = [
    {"consumer": "conclusion", "reads": "identified_arguments",   "on_conclusion_path": True},
    {"consumer": "conclusion", "reads": "identified_fallacies",   "on_conclusion_path": True},
    {"consumer": "conclusion", "reads": "modal_analysis_results", "on_conclusion_path": True},
    {"consumer": "annexe",     "reads": "synth_unread_output",    "on_conclusion_path": False},
]

# Synthetic state: container names mirroring the real surface, PUBLIC
# synthetic content.
TOY_STATE = {
    "identified_arguments": {"arg_1": "Le projet depasse les bornes budgetaires."},
    "identified_fallacies": [{"id": "f_1", "type": "faux dilemme"}],
    "modal_analysis_results": {"valid": True, "formulas": ["[](p -> q)"]},
    "synth_unread_output": {"atoms": ["p", "q"]},
}

# The verdict: which containers weigh on the Act III band. Arguments and
# fallacies carry the band's floor; the modal container is READ for narrative
# but has no entry in the band scale — the "decides-without-weighing" case.
BAND_AXES = {"identified_arguments", "identified_fallacies"}


def verdict(state):
    axes = sum(1 for k in BAND_AXES if state.get(k))
    return "EXCEEDED" if axes >= 2 else ("MET" if axes == 1 else "BELOW")


def classify_all(phases=TOY_PHASES, run=TOY_RUN, state=TOY_STATE,
                 readers=TOY_READERS):
    return {
        p["name"]: journey(p, run, state, readers, verdict) for p in phases
    }


for name, steps in classify_all().items():
    print(f"{name:12s} -> {' -> '.join(steps) if steps else '(jamais appelee)'}")

## 2. Lecture de la matrice

Chaque archétype s'arrête à une étape différente, et **l'étape d'arrêt porte
une information distincte** :

| phase | s'arrête à | ce que ça veut dire |
|---|---|---|
| `p_extract`, `p_fallacy` | **décision** | le verdict changeait si leur sortie changeait — `substantive` |
| `p_formal_a` (miroir du modal) | **lecture** | lue sur le chemin de la conclusion, sans poids sur le verdict |
| `p_formal_b` | **sortie** | écrite dans l'état, aucun lecteur sur le chemin — attesté non mobilisé |
| `p_decor` | **appel** | déclarée dans le run, muette |

L'arrêt n'est pas un jugement moral : une phase `honest-absent` s'arrête aussi
tôt (vide **parce que le corpus manque la structure**, et le dit fail-loud) —
la différence entre `honest-absent` et `theatre` se lit sur la **preuve de
l'arrêt**, pas sur l'arrêt lui-même. C'est la distinction que l'axe verdict
porte, et que l'axe trajet rend vérifiable.

In [ ]:
# Falsifiability — the instrument must have teeth. Three substitutions,
# each MUST change at least one classification, or the classifier is inert
# (the lesson of the gate measurements: a metric invariant between two
# states of the code measures nothing).

# (1) Empty p_extract's output: it must fall back to the appel stop-point.
state_emptied = {k: ({} if k == "identified_arguments" else v)
                 for k, v in TOY_STATE.items()}
before = classify_all(state=TOY_STATE)["p_extract"]
after = classify_all(state=state_emptied)["p_extract"]
assert "sortie" in before and "sortie" not in after, (before, after)
print("(1) sortie vide    : p_extract", before, "->", after)

# (2) Give synth_unread_output an on-path reader (still not in the band):
# it must rise from sortie to lecture — and no further.
readers_plus = TOY_READERS + [
    {"consumer": "conclusion", "reads": "synth_unread_output", "on_conclusion_path": True}
]
before = classify_all()["p_formal_b"]
after = classify_all(readers=readers_plus)["p_formal_b"]
assert before[-1] == "sortie" and after[-1] == "lecture", (before, after)
print("(2) lecteur ajoute : p_formal_b", before, "->", after)

# (3) Now ALSO make it weigh (add to the band), in a state where the band
# can feel it — the output as the ONLY populated container:
BAND_AXES.add("synth_unread_output")
solo_phase = [p for p in TOY_PHASES if p["name"] == "p_formal_b"]
solo = {"synth_unread_output": TOY_STATE["synth_unread_output"]}
in_solo = classify_all(phases=solo_phase, readers=readers_plus, state=solo)["p_formal_b"]
in_empty = classify_all(phases=solo_phase, readers=readers_plus, state={})["p_formal_b"]
assert in_solo[-1] == "decision" and in_empty == ["appel"], (in_solo, in_empty)
print("(3) poids ajoute   : p_formal_b seul present", in_empty, "->", in_solo)

# (4) The threshold caveat, measured on the toy: in the FULL state the same
# phase stays at lecture even with reader AND band weight — because the band
# is a threshold count, already EXCEEDED without it. A band count is
# structurally blind to an axis when richer axes saturate it; saying
# "THIS claim, HERE" needs a per-axis gate, not a band.
full_state = classify_all(readers=readers_plus)["p_formal_b"]
assert full_state[-1] == "lecture", full_state
BAND_AXES.discard("synth_unread_output")
print("(4) bande saturee  : p_formal_b dans l'etat complet reste a", full_state)

print("\nSubstitutions OK — chaque geste deplace la classification qu'il doit "
      "deplacer, et (4) montre ou une bande-compte ne peut PAS sentir un axe.")

## 3. Ce qu'emporte ce notebook

- **L'axe trajet** : quatre étapes ordonnées, chacune prouvée indépendamment
  (phases exécutées / clé peuplée / lecteur vivant sur le chemin / verdict
  sensible à la valeur). Une étape atteinte n'implique pas la suivante.
- **Des dents** : chaque substitution (vider une sortie, ajouter un lecteur,
  ajouter un poids) déplace la classification qu'elle doit déplacer. Un
  classifieur invariant sous substitution ne mesure rien.
- **La borne de la bande** : une bande-compte est structurellement aveugle à
  un axe que des axes plus riches saturent déjà — exprimer « cette
  affirmation-là » exige un gate par axe, pas un seuil de bande.
- **L'honnêteté de l'arrêt** : l'étape où une sortie s'arrête ne dit pas si
  l'arrêt est honnête (`honest-absent`) ou du théâtre — ça, c'est la preuve
  de l'arrêt qui le dit (fail-loud déclarée vs silence vs verdict fabriqué).

**Ce que ce notebook n'est pas** : la mesure. Les classes des 40 phases réelles
vivent dans le fil de l'issue #1605 (mesures historiques datées) et dans les
artefacts de campagne (gitignorés) — la proposition complète, le contrat de
preuve et les créneaux en attente sont dans
`docs/coursia_contrib/richness_matrix_at_scale_note.md`.

> **Références source** (`2025-Epita-Intelligence-Symbolique`) :
> `docs/reports/FP5_FORMAL_RICHNESS_MATRIX.md` (FP-5 #1196),
> `scripts/fp23_measure_unmeasured.py` (FP-23 #1250),
> issue #1605 (matrice à l'échelle, R752/R755/R757).